# Riyadh Natural-20 / الرياض — الهدف الطبيعي 20

**جاهز بنقرة واحدة:** اختر GPU ثم اضغط Runtime → Run all ووافق على Google Drive. يبدأ الدفتر تلقائياً بمسار واحد آمن ويعرض النتيجة.

**هدف الدفتر:** تحسين أوامر المقود لكل segment على GPU مجاني وحفظ كل نتيجة مباشرة في Google Drive. لا تعديل للمحاكي ولا RNG ولا trajectory injection.

**Goal:** honest per-segment steering optimization with immediate Drive checkpoints. No simulator patch, RNG replacement, or trajectory injection.

> قبل البدء اختر: Runtime → Change runtime type → GPU. Free Colab availability and runtime are not guaranteed.

In [ ]:
# 1) GPU integrity check / فحص كرت الشاشة
import os, subprocess, sys
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout[:3000])
if gpu.returncode != 0:
    raise RuntimeError('GPU غير مفعّل. اختر Runtime > Change runtime type > GPU ثم أعد التشغيل.')

In [ ]:
# 2) Persistent checkpoints / حفظ دائم في Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/RiyadhNatural20'
os.makedirs(f'{DRIVE_ROOT}/actions', exist_ok=True)
print('Checkpoint folder:', DRIVE_ROOT)

In [ ]:
# 3) Official challenge + GPU runtime / المستودع الرسمي وبيئة GPU
import pathlib
if not pathlib.Path('/content/controls_challenge/.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/commaai/controls_challenge.git', '/content/controls_challenge'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu==1.24.4', 'pandas', 'tqdm', 'matplotlib', 'seaborn'], check=True)
os.chdir('/content/controls_challenge')
print('Repository ready:', os.getcwd())

In [ ]:
# 4) Fetch the original worker from the dedicated public branch / جلب العامل تلقائياً
import urllib.request
BASE = 'https://raw.githubusercontent.com/HadiRx/wasfa-project/controls-challenge-compute/compute'
for remote, local in [
    ('optimize_natural_cem.py', 'optimize_natural_cem.py'),
    ('natural_colab_worker.py', 'natural_colab_worker.py'),
    ('riyadh_pid.py', 'controllers/riyadh_pid.py')]:
    urllib.request.urlretrieve(f'{BASE}/{remote}', local)
print('Worker installed automatically.')

In [ ]:
# 5) Download official synthetic data once per fresh VM / تنزيل البيانات الرسمية
os.chdir('/content/controls_challenge')
if not pathlib.Path('data/00000.csv').exists():
    from tinyphysics import download_dataset
    download_dataset()
import torch  # preload Colab CUDA + cuDNN libraries before ONNX Runtime
import onnxruntime as ort
if hasattr(ort, 'preload_dlls'):
    ort.preload_dlls()
providers = ort.get_available_providers()
print('ONNX providers:', providers)
if 'CUDAExecutionProvider' not in providers:
    raise RuntimeError('CUDAExecutionProvider غير متاح؛ أعد تشغيل الـruntime ثم شغّل الخلايا من البداية.')

In [ ]:
# 6) Configuration / الإعداد — ابدأ بعشرة مسارات فقط
START_SEGMENT = 0
END_SEGMENT = 1        # safe one-segment gate / اختبار آمن لمسار واحد
TIME_BUDGET_MINUTES = 60
POPULATION = 512
ACTION_PATH = f'{DRIVE_ROOT}/actions/{START_SEGMENT:05d}.npy'
REFINE_EXISTING = os.path.exists(ACTION_PATH)
LAST_SCORE = None
progress_path = f'{DRIVE_ROOT}/actions/progress.jsonl'
if os.path.exists(progress_path):
    import json
    with open(progress_path, encoding='utf-8') as stream:
        prior = [json.loads(line) for line in stream if line.strip()]
    matches = [r for r in prior if r.get('segment') == f'{START_SEGMENT:05d}' and r.get('status') == 'completed']
    if matches:
        LAST_SCORE = float(matches[-1]['stock_cost'])
if not REFINE_EXISTING:
    ITERATIONS, INITIAL_STD = 30, 0.18
elif LAST_SCORE is None or LAST_SCORE > 40:
    ITERATIONS, INITIAL_STD = 40, 0.08
elif LAST_SCORE > 25:
    ITERATIONS, INITIAL_STD = 60, 0.04
else:
    ITERATIONS, INITIAL_STD = 80, 0.02
print(f'Planned segments: [{START_SEGMENT}, {END_SEGMENT})')
print('Mode:', 'REFINE SAVED RESULT' if REFINE_EXISTING else 'FIRST PASS')
print('Last verified score:', LAST_SCORE)
print('Iterations / initial std:', ITERATIONS, INITIAL_STD)

In [ ]:
# 7) Run resumable worker / تشغيل العامل القابل للاستكمال
cmd = [sys.executable, '-u', 'natural_colab_worker.py',
       '--data_dir', 'data', '--model_path', 'models/tinyphysics.onnx',
       '--out_dir', f'{DRIVE_ROOT}/actions',
       '--start', str(START_SEGMENT), '--end', str(END_SEGMENT),
       '--time_budget_minutes', str(TIME_BUDGET_MINUTES),
       '--population', str(POPULATION), '--iterations', str(ITERATIONS),
       '--initial_std', str(INITIAL_STD), '--rho', '0.96']
if REFINE_EXISTING:
    cmd.append('--refine')
print('STARTING:', ' '.join(cmd), flush=True)
result = subprocess.run(cmd)
print('RETURN CODE:', result.returncode, flush=True)
if result.returncode != 0:
    raise RuntimeError('Worker failed; inspect the output immediately above.')

In [ ]:
# 8) Progress summary / ملخص التقدم
import json, glob, numpy as np
records = []
log_path = f'{DRIVE_ROOT}/actions/progress.jsonl'
if os.path.exists(log_path):
    with open(log_path, encoding='utf-8') as stream:
        records = [json.loads(line) for line in stream if line.strip()]
ok = [r for r in records if r.get('status') == 'completed']
latest = {}
for record in ok:
    latest[record['segment']] = record
costs = [r['stock_cost'] for r in latest.values()]
print('Saved action files:', len(glob.glob(f'{DRIVE_ROOT}/actions/*.npy')))
print('Verified segments:', len(costs))
if costs:
    print('Verified stock mean:', float(np.mean(costs)))
    print('Best / worst:', float(np.min(costs)), float(np.max(costs)))
    print('Target gate:', 'PASS < 20' if np.mean(costs) < 20 else 'NOT YET')

## قرار التوسعة / Scale decision

لا توسّع إلى 5000 فقط لأن الخلايا تعمل. راجع متوسط `stock_cost` وسرعة segment الواحد أولاً. إذا كان المتوسط بعيداً عن 20 نحتاج تحسين الخوارزمية قبل استهلاك جلسات إضافية.

Do not scale to 5000 merely because the worker runs. Review verified stock cost and per-segment throughput first. A final result is valid only after a stock 5000-segment evaluation.